# E1 — Warm-start RL: does RL help *after* CE?

Three arms from the released CE checkpoint
(`esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b`, written literally in every
command — no notebook variables): W1 warm+REINFORCE, W2 warm+KL-REINFORCE
(coef swept on seed 7), W3 warm+exact-PG.

Plain `!` commands, same style as the notebooks that worked — output
streams live (`python -u` = unbuffered; an eval line prints every 25
steps). Runtime → T4/L4 GPU → Run all.
Download `warmstart_results.zip` at the end.

In [ ]:
import os
os.chdir('/content')
if not os.path.isdir('verifier-as-reward'):
    !git clone https://github.com/esmaeil-abedi-dev/verifier-as-reward.git
os.chdir('/content/verifier-as-reward')
!git pull --ff-only
import torch; assert torch.cuda.is_available(), 'enable a GPU runtime'
!PYTHONPATH=. python test_authority_verifier.py
!PYTHONPATH=. python make_expanded_train.py --train-seed 101 --train-traces-per-class 150 --val-seed 202 --val-traces-per-class 25

## Step 1 — KL sweep on seed 7 (~45 min: three 500-step runs)
An eval line prints every 25 steps. Model download + load takes ~1–2 min
at the start of each run.

In [ ]:
!for kl in 0.02 0.1 0.5; do echo "=== KL=$kl ==="; PYTHONPATH=. python -u train_verifier_reward.py --model Qwen/Qwen2.5-0.5B --warm-start-from esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b --kl-coef $kl --balance-reward --prompt-style nl --steps 500 --batch-size 16 --lr 2e-5 --train-file expanded_train.jsonl --test-file expanded_val.jsonl --eval-every 25 --eval-max-actions 120 --seed 7 --log-file training_log_warmstart_klsweep_$kl.jsonl || break; done

In [ ]:
import json
for kl in ['0.02', '0.1', '0.5']:
    pts = [json.loads(l) for l in open(f'training_log_warmstart_klsweep_{kl}.jsonl') if '"step"' in l]
    e = pts[-1]
    print(f"KL={kl}: final val acc {e['accuracy']:.3f} viol {e['heldout_violation_rate']} fref {e['heldout_false_refuse_rate']}")
print()
print('Pick the best KL (highest acc, both errors low). The W2 cell below')
print('uses 0.1 written literally — edit that cell if the sweep disagrees.')

## Step 2 — three arms x three seeds (~2.5 h total)

In [ ]:
!for s in 7 8 9; do echo "=== w1_reinforce seed $s ==="; PYTHONPATH=. python -u train_verifier_reward.py --model Qwen/Qwen2.5-0.5B --warm-start-from esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b --balance-reward --prompt-style nl --steps 500 --batch-size 16 --lr 2e-5 --train-file expanded_train.jsonl --test-file expanded_val.jsonl --eval-every 25 --eval-max-actions 120 --seed $s --log-file training_log_warmstart_w1_reinforce_seed$s.jsonl --save-dir ckpt_warmstart_w1_reinforce_seed$s || break; done

In [ ]:
# kl-coef 0.1 below — edit if the Step-1 sweep chose a different value
!for s in 7 8 9; do echo "=== w2_kl seed $s ==="; PYTHONPATH=. python -u train_verifier_reward.py --model Qwen/Qwen2.5-0.5B --warm-start-from esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b --kl-coef 0.1 --balance-reward --prompt-style nl --steps 500 --batch-size 16 --lr 2e-5 --train-file expanded_train.jsonl --test-file expanded_val.jsonl --eval-every 25 --eval-max-actions 120 --seed $s --log-file training_log_warmstart_w2_kl_seed$s.jsonl --save-dir ckpt_warmstart_w2_kl_seed$s || break; done

In [ ]:
!for s in 7 8 9; do echo "=== w3_exactpg seed $s ==="; PYTHONPATH=. python -u train_verifier_reward.py --model Qwen/Qwen2.5-0.5B --warm-start-from esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b --exact-pg --balance-reward --prompt-style nl --steps 500 --batch-size 16 --lr 2e-5 --train-file expanded_train.jsonl --test-file expanded_val.jsonl --eval-every 25 --eval-max-actions 120 --seed $s --log-file training_log_warmstart_w3_exactpg_seed$s.jsonl --save-dir ckpt_warmstart_w3_exactpg_seed$s || break; done

## Step 3 — evaluate everything vs the CE start (committed test + fresh set)

In [ ]:
# Fresh in-distribution set (new seed, deduped vs train) for a tighter CI.
from trace_benchmark import generate_corpus, write_jsonl, load_traces
from make_expanded_train import corpus_canonicals, drop_overlapping
test_canon = corpus_canonicals(load_traces('benchmark_test.jsonl'))
a, b = generate_corpus(101, 150)
train_canon = corpus_canonicals(drop_overlapping(a + b, test_canon, 'train'))
c, d = generate_corpus(770, 60)
fresh = drop_overlapping(c + d, train_canon | test_canon, 'fresh')
write_jsonl(fresh, 'warmstart_fresh.jsonl')
print('fresh actions:', sum(len(t['actions']) for t in fresh))
import json; json.dump({'backends': {}}, open('results_warmstart.json', 'w'))

In [ ]:
# Baseline: the CE start itself, on both sets.
!PYTHONPATH=. python train_verifier_reward.py --eval-checkpoint esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b --test-file benchmark_test.jsonl --merge-results results_warmstart.json
!PYTHONPATH=. python train_verifier_reward.py --eval-checkpoint esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b --test-file warmstart_fresh.jsonl --merge-results results_warmstart.json

In [ ]:
# Every trained arm on both sets (skips arms without a checkpoint).
!for arm in w1_reinforce w2_kl w3_exactpg; do for s in 7 8 9; do ck=ckpt_warmstart_${arm}_seed$s; if [ -d $ck ]; then for tf in benchmark_test.jsonl warmstart_fresh.jsonl; do echo "=== $ck on $tf ==="; PYTHONPATH=. python train_verifier_reward.py --eval-checkpoint $ck --test-file $tf --merge-results results_warmstart.json; done; fi; done; done

In [ ]:
!zip -j warmstart_results.zip training_log_warmstart_*.jsonl results_warmstart.json
from google.colab import files
files.download('warmstart_results.zip')